In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import (accuracy_score,f1_score,roc_auc_score,confusion_matrix,classification_report)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
!pip install xgboost
from xgboost import XGBClassifier


from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
TW = pd.read_csv(
    r"../classification/Twitter/Absolute_labeling/Twitter-Absolute-Sigma-500.data",
    sep=",",
    header=None
)

groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TW.columns = columns

TW.head(2)

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,AI_0,AI_1,AI_2,...,ADL_5,ADL_6,NAD_0,NAD_1,NAD_2,NAD_3,NAD_4,NAD_5,NAD_6,label
0,889,939,960,805,805,1143,1121,549,613,587,...,1.0,1.0,889,939,960,805,805,1143,1121,1.0
1,542,473,504,626,647,795,832,366,288,318,...,1.0,1.0,542,473,504,626,647,795,832,1.0


In [3]:
TH = pd.read_csv(
    "../classification/TomsHardware/Absolute_labeling/TomsHardware-Absolute-Sigma-500.data",
    sep=",",
    header=None
)


groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TH.columns = columns


prefixes = {col.split("_")[0] for col in TH.columns if "_" in col}


TH.head()

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,NCD_7,BL_0,BL_1,...,AS_NA_7,AS_NAC_0,AS_NAC_1,AS_NAC_2,AS_NAC_3,AS_NAC_4,AS_NAC_5,AS_NAC_6,AS_NAC_7,label
0,1,0,0,0,0,0,0,1,1.0,0.0,...,0.001816,0.001211,0.000560,0.000000,0.000000,0.000161,0.0,0.000301,0.000818,1.0
1,1,1,1,1,0,0,0,0,1.0,1.0,...,0.005029,0.000784,0.000802,0.001592,0.001612,0.000741,0.0,0.000545,0.002437,1.0
2,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
3,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
4,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0


### Twitter

In [6]:
X = TW.drop(columns=["label"])
y = TW["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### Decision Tree baseline

In [9]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)
pred_proba = dt.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_dt)
precision = precision_score(y_test, pred_dt, zero_division=0)
recall = recall_score(y_test, pred_dt, zero_division=0)
f1 = f1_score(y_test, pred_dt, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Decision Tree baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Decision Tree baseline
Accuracy: 0.9496482126359178
Precision: 0.8729271809661139
Recall: 0.8718271827182719
F1-score: 0.8723768350896154
ROC-AUC: 0.9203072691383896


### Decision Tree with class_weight

In [10]:
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)
pred_proba = dt.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_dt)
precision = precision_score(y_test, pred_dt, zero_division=0)
recall = recall_score(y_test, pred_dt, zero_division=0)
f1 = f1_score(y_test, pred_dt, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Decision Tree with class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Decision Tree with class_weight
Accuracy: 0.9498258830218179
Precision: 0.8765678967460462
Recall: 0.868046804680468
F1-score: 0.872286541244573
ROC-AUC: 0.9189926324283377


### Decision Tree with Grid Search

In [13]:
param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=42),param_grid,cv=5,scoring='f1',n_jobs=-1)
grid.fit(X_train, y_train)

best_dt = grid.best_estimator_
pred_dt = best_dt.predict(X_test)
pred_proba = best_dt.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_dt)
precision = precision_score(y_test, pred_dt, zero_division=0)
recall = recall_score(y_test, pred_dt, zero_division=0)
f1 = f1_score(y_test, pred_dt, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Decision Tree with Grid Search")
print("Best params:", grid.best_params_)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Decision Tree with Grid Search
Best params: {'class_weight': None, 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}
Accuracy: 0.9653542747494848
Precision: 0.9309371471584493
Recall: 0.8905490549054905
F1-score: 0.9102953353574386
ROC-AUC: 0.9910109831543653


In [20]:
import pandas as pd

data = {
    "Model": [
        "Decision Tree",
        "Decision Tree with class_weight",
        "Decision Tree with Grid Search"
    ],
    "Accuracy": [0.9496, 0.9498, 0.9654],
    "Precision": [0.8729, 0.8766, 0.9309],
    "Recall": [0.8718, 0.8680, 0.8905],
    "F1-score": [0.8724, 0.8723, 0.9103],
    "ROC-AUC": [0.9203, 0.9190, 0.9910]
}

df_dt_twitter = pd.DataFrame(data)
df_dt_twitter

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Decision Tree,0.9496,0.8729,0.8718,0.8724,0.9203
1,Decision Tree with class_weight,0.9498,0.8766,0.8680,0.8723,0.9190
2,Decision Tree with Grid Search,0.9654,0.9309,0.8905,0.9103,0.9910


###  Tom Hardware

In [14]:
X = TH.drop(columns=["label"])
y = TH["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### Decision Tree baseline

In [15]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)
pred_proba = dt.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_dt)
precision = precision_score(y_test, pred_dt, zero_division=0)
recall = recall_score(y_test, pred_dt, zero_division=0)
f1 = f1_score(y_test, pred_dt, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Decision Tree baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Decision Tree baseline
Accuracy: 0.9576217583807717
Precision: 0.9718456725755996
Recall: 0.9588477366255144
F1-score: 0.9653029518384257
ROC-AUC: 0.9572563806280281


### Decision Tree with class_weight

In [16]:
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)
pred_proba = dt.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_dt)
precision = precision_score(y_test, pred_dt, zero_division=0)
recall = recall_score(y_test, pred_dt, zero_division=0)
f1 = f1_score(y_test, pred_dt, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Decision Tree with class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Decision Tree with class_weight
Accuracy: 0.9607843137254902
Precision: 0.9739583333333334
Recall: 0.9619341563786008
F1-score: 0.9679089026915114
ROC-AUC: 0.9604416266293662


### Decision Tree with Grid Search

In [17]:
param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=42),param_grid,cv=5,scoring='f1',n_jobs=-1)
grid.fit(X_train, y_train)

best_dt = grid.best_estimator_
pred_dt = best_dt.predict(X_test)
pred_proba = best_dt.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_dt)
precision = precision_score(y_test, pred_dt, zero_division=0)
recall = recall_score(y_test, pred_dt, zero_division=0)
f1 = f1_score(y_test, pred_dt, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Decision Tree with Grid Search")
print("Best params:", grid.best_params_)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Decision Tree with Grid Search
Best params: {'class_weight': None, 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}
Accuracy: 0.963314358001265
Precision: 0.9760416666666667
Recall: 0.9639917695473251
F1-score: 0.9699792960662525
ROC-AUC: 0.9898825572516505


In [22]:
data = {
    "Model": [
        "Decision Tree",
        "Decision Tree with class_weight",
        "Decision Tree Grid Search"
    ],
    "Accuracy": [0.9576, 0.9608, 0.9633],
    "Precision": [0.9718, 0.9740, 0.9760],
    "Recall": [0.9588, 0.9619, 0.9640],
    "F1-score": [0.9653, 0.9679, 0.9700],
    "ROC-AUC": [0.9573, 0.9604, 0.9899]
}

df_dt_th = pd.DataFrame(data)
df_dt_th

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Decision Tree,0.9576,0.9718,0.9588,0.9653,0.9573
1,Decision Tree with class_weight,0.9608,0.9740,0.9619,0.9679,0.9604
2,Decision Tree Grid Search,0.9633,0.9760,0.9640,0.9700,0.9899
